# To generate P3 orderings reported in the paper
### PHYSICAL REVIEW MATERIALS 3, 015402 (2019)
### PHYSICAL REVIEW MATERIALS 6, 115401 (2022)

In [13]:
import numpy as np
from pymatgen.core import Structure, Lattice

def _as_frac(struct, atom):
    """Accept int index or (3,) fractional array; return fractional coord."""
    if isinstance(atom, (int, np.integer)):
        return np.array(struct.frac_coords[int(atom)], float)
    a = np.array(atom, float)
    if a.shape != (3,):
        raise ValueError("Atom must be index or 3-vector.")
    return a

def _wrap01(x):
    """Wrap fractional coords into [0,1)."""
    return x - np.floor(x)

def _snap_to_grid(v, grid=None, axes=(0,1)):
    """Snap selected components of v to multiples of 1/grid."""
    if grid is None:
        return v
    v = v.copy()
    for ax in axes:
        v[ax] = np.round(v[ax] * grid) / float(grid)
    return v

def _periodic_bin_key(fc, nbin, axes=(0,1,2)):
    """
    Map fractional coord to an integer bin key with periodic wrap.
    Nearest-bin quantization + modulo so 0.999.. and 0.0 collide.
    """
    key = []
    for ax in axes:
        # nearest bin index in [0, nbin-1]
        k = int(np.round(fc[ax] * nbin)) % nbin
        key.append(k)
    return tuple(key)

def cut_cell_from_four_atoms_large(
    struct: Structure,
    A, B, C, D=None,
    keep_c=True,
    grid=None,
    shifts=((0,0,0), (0,0,0)),   # integer image shifts to enlarge vectors
    enforce_right_handed=True,
    dedup_tol=1e-4,
):
    """
    Build a new unit cell spanned by vectors A->B and A->C (optionally enlarged by integer
    image shifts). NO nearest-image wrapping is applied.

    Parameters
    ----------
    struct : Structure (e.g., a 7x7 supercell of a single Na layer)
    A,B,C,D : int indices or fractional 3-vectors (D optional consistency check)
    keep_c : keep old c-axis as v3
    grid : int; snap in-plane components to multiples of 1/grid (use 7 for a 7x7 mesh)
    shifts : tuple of two integer 3-tuples added to (B-A) and (C-A). Example:
             shifts=((+1,0,0),(0,+2,0)) makes A->B one a-lattice longer, A->C two b longer.
    enforce_right_handed : if True, swap v1,v2 to make det(B)>0
    dedup_tol : merge coincident atoms after folding

    Returns
    -------
    new_struct : Structure
    info : dict with B, B_inv, origin shift, determinant, and D-closure error.
    """
    fA = _as_frac(struct, A)
    fB = _as_frac(struct, B)
    fC = _as_frac(struct, C)
    print(fA, fB, fC)

    # Use raw differences + user-chosen image shifts (integers)
    m1 = np.array(shifts[0], dtype=float)
    m2 = np.array(shifts[1], dtype=float)
    v1 = (fB - fA) + m1
    v2 = (fC - fA) + m2

    # Snap only in-plane components to the commensurate grid if provided
    v1 = _snap_to_grid(v1, grid, axes=(0,1))
    v2 = _snap_to_grid(v2, grid, axes=(0,1))

    # Third vector
    v3 = np.array([0.0, 0.0, 1.0]) if keep_c else np.array([0.0, 0.0, 1.0])

    # Optional consistency check with the 4th corner
    predicted_D_error = None
    if D is not None:
        fD = _as_frac(struct, D)
        pred = fA + v1 + v2
        # error modulo integers
        delta = fD - pred - np.round(fD - pred)
        predicted_D_error = float(np.linalg.norm(delta))

    # Basis-change matrix (rows are new basis vectors in old fractional coords)
    B = np.vstack([v1, v2, v3])
    print(B)
    detB = float(np.linalg.det(B))
    if abs(detB) < 1e-10:
        raise ValueError("New basis is singular. Choose independent A→B and A→C.")

    if enforce_right_handed and detB < 0:
        v1, v2 = v2, v1
        B = np.vstack([v1, v2, v3])
        detB = float(np.linalg.det(B))

    B_inv = np.linalg.inv(B)

    # Build new lattice: L_new = B @ L_old
    L_old = struct.lattice.matrix
    L_new = B @ L_old
    lat_new = Lattice(L_new)

    # Transform all sites to new basis with origin at A, then wrap into [0,1)
    # For P3 case, maintain z-coordinates as-is.
    fA_prime = [fA[0], fA[1], 0.0]
    f_old = struct.frac_coords
    f_new = ((f_old - fA_prime) @ B_inv)
    f_new = _wrap01(f_new)

    # ---------- PERIODIC-AWARE DEDUP ----------
    # Choose number of bins so that one bin ~ dedup_tol
    # (cap extreme cases so nbin doesn't explode numerically)
    nbin = max(2, int(np.round(1.0 / max(dedup_tol, 1e-8))))
    seen = {}
    species_kept, coords_kept = [], []
    for sp, fc in zip(struct.species, f_new):
        key_coords = _periodic_bin_key(fc, nbin)
        key = key_coords
        if key in seen:
            continue
        seen[key] = True
        species_kept.append(sp)
        # store a canonical coord: round to bin center to stabilize
        fc_can = fc.copy()
        for i, ax in enumerate([0,1,2]):
            # bin center = k / nbin (since unit width is 1/nbin)
            k = key_coords[i]
            fc_can[ax] = (k % nbin) / float(nbin)
        coords_kept.append(_wrap01(fc_can))

    new_struct = Structure(lat_new, species_kept, np.array(coords_kept), coords_are_cartesian=False)

    info = dict(
        B=B, B_inv=B_inv, detB=detB,
        origin_shift_old_frac=fA,
        predicted_D_error=predicted_D_error,
        n_sites_in=len(struct), n_sites_out=len(new_struct),
        v1=v1, v2=v2, v3=v3,
        shifts=shifts, grid=grid,
        dedup_tol=dedup_tol, nbin=nbin
    )
    return new_struct, info

In [30]:
from pymatgen.core import Structure

s = Structure.from_file("../data/P3_orderings/P3_NaCoO2_771.vasp")  # your 7×7 supercell

# Suppose the four Na you want are iA,iB,iC,iD (replace with your indices)
iA, iB, iC, iD = 205, 225, 214, 234

s_big, info = cut_cell_from_four_atoms_large(
    s, iA, iB, iC, D=iD,
    grid=7,
    shifts=((0,0,0), (0,0,0)),  # enlarge on purpose
    keep_c=True
)

print("det(B) =", info["detB"], " sites:", info["n_sites_in"], "->", info["n_sites_out"])
s_big.to(fmt="poscar", filename="../data/P3_orderings/JLK_zeta3.vasp")

[0.19047619 0.38095238 0.83564167] [0.61904762 0.23809524 0.83564167] [0.33333333 0.66666667 0.83564167]
[[ 0.42857143 -0.14285714  0.        ]
 [ 0.14285714  0.28571429  0.        ]
 [ 0.          0.          1.        ]]
det(B) = 0.14285714285714285  sites: 735 -> 105


'Na42 Co21 O42\n1.0\n   3.0191004616857144  -10.4584707855999994    0.0000000000000000\n   4.5286506925285712    2.6146176963999994    0.0000000000000000\n   0.0000000000000000    0.0000000000000000   16.4176330565999997\nNa Co O\n42 21 42\ndirect\n   0.0000000000000000    0.6667000000000000    0.1690000000000000 Na\n   0.8571000000000000    0.0952000000000000    0.1690000000000000 Na\n   0.7143000000000000    0.5238000000000000    0.1690000000000000 Na\n   0.5714000000000000    0.9524000000000000    0.1690000000000000 Na\n   0.4286000000000000    0.3810000000000000    0.1690000000000000 Na\n   0.2857000000000000    0.8095000000000000    0.1690000000000000 Na\n   0.1429000000000000    0.2381000000000000    0.1690000000000000 Na\n   0.1429000000000000    0.9048000000000000    0.1690000000000000 Na\n   0.0000000000000000    0.3333000000000000    0.1690000000000000 Na\n   0.8571000000000000    0.7619000000000000    0.1690000000000000 Na\n   0.7143000000000000    0.1905000000000000    0.16

In [20]:
iD

24